In [1]:
import pandas as pd
import ast
import json
from pathlib import Path

# ================= 配置区 =================

ALL_TSV = "02_source_data/data/all.tsv"
CLASSES_TSV = "02_source_data/data/classes.tsv"

IMAGE_DIR = "data/images"              # 图像所在目录
OUTPUT_JSONL = "odil_timel_sft.jsonl"

# 冻结的法语 prompt（只作为占位符）
PROMPT_TEXT = "Générer les identifiants timel pour cette image."

# ================= Step 1: 读取 classes.tsv → timel 白名单 =================

classes_df = pd.read_csv(CLASSES_TSV, sep="\t")

if "timel_id" not in classes_df.columns:
    raise ValueError("classes.tsv 必须包含列: timel_id")

timel_vocab = set(classes_df["timel_id"].astype(str))

print(f"[INFO] timel vocab size = {len(timel_vocab)}")

# ================= Step 2: 读取 all.tsv =================

all_df = pd.read_csv(ALL_TSV, sep="\t")

required_cols = {"Image", "timel_ids_list"}
missing = required_cols - set(all_df.columns)
if missing:
    raise ValueError(f"all.tsv 缺少列: {missing}")

# ================= Step 3: 构造 SFT 样本 =================

samples = []
dropped = 0

for _, row in all_df.iterrows():
    image_name = str(row["Image"]).strip()

    # 解析 timel_ids_list（字符串 → Python list）
    try:
        timel_ids = ast.literal_eval(row["timel_ids_list"])
    except Exception:
        dropped += 1
        continue

    if not isinstance(timel_ids, list):
        dropped += 1
        continue

    # 白名单过滤
    kept = sorted(t for t in timel_ids if t in timel_vocab)

    if not kept:
        dropped += 1
        continue

    image_path = str(Path(IMAGE_DIR) / image_name)

    # assistant 输出：标签列表（逗号分隔）
    label_text = ", ".join(kept)

    sample = {
        "images": [image_path],
        "messages": [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": image_path},
                    {"type": "text", "text": PROMPT_TEXT}
                ]
            },
            {
                "role": "assistant",
                "content": [
                    {"type": "text", "text": label_text}
                ]
            }
        ]
    }

    samples.append(sample)

print(f"[INFO] kept samples   = {len(samples)}")
print(f"[INFO] dropped samples = {dropped}")

# ================= Step 4: 写出 JSONL =================

with open(OUTPUT_JSONL, "w", encoding="utf-8") as f:
    for s in samples:
        f.write(json.dumps(s, ensure_ascii=False) + "\n")

print(f"[DONE] 写出完成: {OUTPUT_JSONL}")

[INFO] timel vocab size = 2333
[INFO] kept samples   = 9132
[INFO] dropped samples = 0
[DONE] 写出完成: odil_timel_sft.jsonl


In [2]:
import json

with open("odil_timel_sft.jsonl", encoding="utf-8") as f:
    sample = json.loads(next(f))

print(sample["messages"][0]["content"][1]["text"])  # user prompt
print(sample["messages"][1]["content"][0]["text"])  # assistant labels

Générer les identifiants timel pour cette image.
tm-6dqqjsez, tm-7schjv64, tm-9kefvmxz, tm-ec7e7dqn, tm-f4sqsvcv, tm-fmytavcm, tm-hwbpfzqq, tm-kancnlxz, tm-pphjzwxj, tm-sz7srifg, tm-wqg2nktt, tm-zn7lmvzm


In [3]:
import json
import random

INPUT_JSONL = "odil_timel_sft.jsonl"
TRAIN_JSONL = "train.jsonl"
VAL_JSONL = "val.jsonl"

VAL_RATIO = 0.05   # 5% 验证集
SEED = 42

random.seed(SEED)

lines = []
with open(INPUT_JSONL, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            lines.append(line)

random.shuffle(lines)

n_val = int(len(lines) * VAL_RATIO)
val_lines = lines[:n_val]
train_lines = lines[n_val:]

with open(TRAIN_JSONL, "w", encoding="utf-8") as f:
    f.write("\n".join(train_lines) + "\n")

with open(VAL_JSONL, "w", encoding="utf-8") as f:
    f.write("\n".join(val_lines) + "\n")

print("total =", len(lines))
print("train =", len(train_lines))
print("val   =", len(val_lines))

total = 9132
train = 8676
val   = 456


In [4]:
import json
from pathlib import Path

JSONL = "train.jsonl"
N = 50

missing = 0
with open(JSONL, "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i >= N:
            break
        obj = json.loads(line)
        img = obj["images"][0]
        if not Path(img).exists():
            missing += 1
            print("missing:", img)

print("missing in first", N, "=", missing)

missing in first 50 = 0
